# Train the LeJEPA encoder on OGBench cube-single, and score what it recovered

One encoder, one seed, and the frozen metric suite run on the **last few
checkpoints**. The question it answers is *"how much of the true latent state
did this encoder recover, and which coordinates did it miss?"*

Prerequisite: `collect_lejepa_ogbcubesingle.ipynb` has pushed both datasets — the
OU pairs **and** the style probe — to HF.

Flow: config → apply → install → download → W&B → pre-flight → smoke → train → **statistics** → read it.

## Where the statistics live

This notebook holds the **encoder-side** statistics: `run_metrics.py`, run on
the last two or three checkpoints. They belong here because they need a
checkpoint.

**Not once per epoch.** Each invocation embeds 20k pairs twice over, embeds the
style probe, and trains a non-linear probe — minutes per checkpoint, and it buys
nothing on a flat stretch. The training run already logs a per-step trend
(`recovery/*`, `spectrum/*`); this is the authoritative measurement, taken where
it matters.

The **dataset-side** statistics (the marginal, the achieved ρ, and the
render round-trip) ran in the collection notebook, before the upload.

## What one seed buys, and what it does not

With `n = 1` seed no metric here carries an error bar, so **read the per-latent
breakdown, not the aggregate at the last epoch**. The results rows record `seed`
and `program_constants` verbatim, so a later multi-seed run appends to the same
file and the two are directly comparable.

## 1. Config

In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ land directly here
SPT_CACHE_DIR = '/workspace/cache/stable-pretraining'  # Lightning .ckpt files, not used by the metrics

# --- Hugging Face ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')                                # ← paste if not a pod env var
HF_REPO_ID = 'quastAI/lejepa-ogbench-cube-single-ou'   # ← from the collection notebook

# --- Weights & Biases ---
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')  # ← paste if not a pod env var
WANDB_ENTITY = 'julian-quast-8-technical-university-of-berlin'                 # ← edit me
WANDB_PROJECT = 'lejepa-identifiability'             # launcher/local.yaml would otherwise force 'stable-wm'

# --- the run ---
PROFILE = 'physical_content'   # must match the dataset that was collected
ENCODER = 'paper_cnn'          # the paper's own pixel encoder; `vit_small` is the alternative
# RUN THIS TO COMPLETION. At CONSTANT_FRAC=0.5 the whole first half is at peak
# lr, so killing anywhere in it forfeits the entire anneal -- and the anneal is
# where the alignment loss actually approaches its floor. There is no resume
# (`ckpt_path=None` in lejepa.py), so a killed run is not a shorter run, it is a
# checkpoint frozen at peak lr. 100 epochs x 703 steps = 70k steps, ~1.8x the
# paper's own Reacher budget, so under-training stops being an available
# explanation for whatever comes back; 55 epochs would match their step count
# exactly if an epoch turns out to cost more than ~10 minutes.
# `SaveCkptCallback` still writes weights_epoch_N.pt every epoch, so a crashed
# run keeps everything up to the crash.
# 50, not 100, for the first run. Because `constant_frac` rescales with the
# budget, a 50-epoch run is NOT a truncated 100 -- it holds at peak for 25
# epochs and anneals over 25, so it is a complete, fully-annealed experiment in
# its own right. 50 x 703 = 35,150 steps is 90% of the paper's own Reacher
# budget, which is already enough to retire "under-trained" as an explanation.
# Go to 100 only if recovery is still climbing through the final anneal epochs;
# the two runs' INTERMEDIATE checkpoints are not comparable to each other,
# since each sits at its own lr in a differently-scaled anneal.
EPOCHS = 50
# Fraction of the run held at PEAK lr before the cosine anneal to zero begins.
# 0.5 is the paper's recipe (App. H.4, "constant for the first half of training,
# followed by cosine decay to zero"); 0.0 is a pure cosine; 1.0 is flat forever.
#
# The hold is what buys representation learning, the anneal is what makes the
# final number mean anything. A constant lr does not converge, it orbits in a
# noise ball whose radius scales with the lr, and `align_loss` is a difference
# of the two views' embeddings -- exactly what a jittering encoder inflates. The
# s3072 25-epoch constant run plateaued at 2.29x its alignment floor where the
# paper's annealed runs sit at 0.976x, so THAT PLATEAU WAS NOT A CEILING. A pure
# cosine has the opposite problem: lr is already at half peak by the halfway
# point, so a big EPOCHS mostly buys anneal rather than learning.
CONSTANT_FRAC = 0.5
# Absolute, no longer 1% of total steps. The s3072 run got 70 warmup steps
# because it was a 10-epoch run; 700 is the 100-epoch schedule's own value.
WARMUP_STEPS = 700
SEED = 3072
BATCH_SIZE = 256
NUM_WORKERS = 6
LAMBDA = 5.0e-2                # loss = lambda*SIGReg + (1-lambda)*align, in the PAPER's units
OUTPUT_MODEL_NAME = 'lejepa'

# --- metrics ---
# 20000 is the suite's default. Each scoring embeds 2 views x (OU set + style
# probe) and then trains the non-linear probe, so this is the knob that decides
# how long §10 takes.
METRIC_MAX_SAMPLES = 20_000
# How many of the LAST saved checkpoints to score. Scoring every epoch is what
# this notebook used to do and it is not affordable: minutes per checkpoint,
# spent almost entirely on a flat stretch. The training run logs a per-step
# trend already (`recovery/*`, `spectrum/*`); this suite is the authoritative
# reading, and 2-3 points at the end is enough to see whether it is still
# moving. Under 'cosine' these last points are also the ONLY settled ones --
# earlier epochs each sit at their own lr. Raise it only for a stretch you
# actually intend to inspect.
SCORE_LAST_N = 3
# Refresh BatchNorm running statistics over the eval set before scoring, with
# the weights frozen. ON by default -- and this is not cosmetic: BN's default
# momentum=0.1 is a ~10-batch window, so the statistics any mid-run checkpoint
# carries are both noisy and stale relative to its own weights. Every number the suite reports is computed through
# them. `bn_recalibrated` is recorded per row; set this False only to measure
# the size of the effect, and expect the two rows to differ. The anneal fixes
# the staleness at the source for the FINAL checkpoint only; every earlier one
# is still taken mid-motion, so this stays on.
RECALIBRATE_BN = True

# --- supervised ceiling (section 10b) ---
# Trains the SAME architecture on the SAME pixels with the labels handed to it,
# which is the only thing in this notebook that can tell "LeJEPA missed it"
# from "it was never in the pixels". Cheap relative to training: supervised
# regression converges in a fraction of the SSL budget.
#
# An UNDER-trained oracle reports a ceiling that is too LOW, which is the
# dangerous direction -- it would excuse a real LeJEPA failure as
# unobservability. If the per-latent scores are still climbing at the end,
# raise this rather than trusting the number.
ORACLE_EPOCHS = 20
ORACLE_MAX_SAMPLES = 20_000

# --- derived, don't edit ---
SUBDIR = f'{OUTPUT_MODEL_NAME}_s{SEED}'
OGBENCH_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench')
CKPT_DIR = os.path.join(STABLEWM_HOME, 'checkpoints', OUTPUT_MODEL_NAME)
OUTPUT_DIR = os.path.join(STABLEWM_HOME, 'outputs')
RESULTS_PATH = os.path.join(OUTPUT_DIR, f'lejepa_results_{SUBDIR}.jsonl')
OU_NAME = f'ogbench/cube_single_ou_{PROFILE}.lance'
STYLE_NAME = f'ogbench/cube_single_ou_style_{PROFILE}.lance'

## 2. Apply config

In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['SPT_CACHE_DIR'] = SPT_CACHE_DIR
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

# Many-core pods (e.g. EPYC) default OpenBLAS/MKL to one thread pool per
# process for every small linalg call (run_metrics.py's identifiability
# suite) -- on tiny n=10 matrices the thread-spawn overhead
# dwarfs the FLOPs, so wall-clock can run 10-20x behind CPU-seconds consumed
# (diagnosed live: 63 CPU-min inside 20 wall-min on a 64-core pod). Pin to one
# thread per BLAS call; PYTHONUNBUFFERED so a piped subprocess's prints (loguru
# excluded, it flushes on its own) don't sit in a block buffer until it exits.
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

for path in (STABLEWM_HOME, SPT_CACHE_DIR, OGBENCH_DIR, OUTPUT_DIR):
    os.makedirs(path, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it behaves the same after a kernel restart

print('cwd           =', os.getcwd())
print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])
print('checkpoints ->', CKPT_DIR)
print('results     ->', RESULTS_PATH)
!df -h "$STABLEWM_HOME"

## 3. Torch ≥ 2.5

`transformers` needs `torch>=2.5`; some pods ship 2.4.1. **Restart the kernel if it
upgrades**, then re-run cells 1–2.

In [ ]:
import torch
print('torch before:', torch.__version__)

if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 5):
    !pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('Upgraded — RESTART THE KERNEL now, then re-run cells 1-2 before continuing.')
else:
    print('torch already >= 2.5, nothing to do.')

## 4. Install dependencies

In [ ]:
%pip install -q -e '.[train,format]' wandb huggingface_hub

## 5. Download the datasets

Both tables plus both sidecar manifests, straight into `$STABLEWM_HOME/datasets/ogbench/`.
The manifests are not optional: `run_metrics.py` **aborts** on a missing one, because it
reads ρ, the violation key, `config_hash`, `latents.n` and the isotropy margin from there
and guessing a default is how two runs become indistinguishable.

Safe to re-run.

In [ ]:
!hf download "$HF_REPO_ID" --repo-type dataset --local-dir "$OGBENCH_DIR"

In [ ]:
import json
from pathlib import Path

import lance

for name in (OU_NAME, STYLE_NAME):
    stem = Path(name).stem
    table = Path(OGBENCH_DIR) / f'{stem}.lance'
    sidecar = Path(OGBENCH_DIR) / f'{stem}_manifest.json'
    assert table.exists(), f'missing dataset {table}'
    assert sidecar.exists(), f'missing manifest {sidecar} — run_metrics.py will abort'
    manifest = json.loads(sidecar.read_text())
    rows = lance.dataset(str(table)).count_rows()
    print(f'{stem:44s} {rows // 2:>8,} pairs  n={manifest["latents"]["n"]:<3d} '
          f'rho={manifest["ou"]["rho_mean"]}  hash={manifest["config_hash"]}')

N_LATENT = json.loads((Path(OGBENCH_DIR) / f'{Path(OU_NAME).stem}_manifest.json').read_text())['latents']['n']
print(f'\nhead width m will be taken from the data: m = n = {N_LATENT}')

## 6. W&B login

In [ ]:
import wandb

wandb.login(key=WANDB_API_KEY)

## 7. Pre-flight

Two gates, both cheap, both before any GPU time is spent.

In [ ]:
assert torch.cuda.is_available(), 'No GPU visible — lejepa.yaml pins accelerator: gpu'
print(torch.cuda.get_device_name(0))
print('devices pinned to 1 on purpose: SIGReg is a batch statistic and is never gathered '
      'across ranks, so two devices would silently halve the effective batch.')

A small helper for the subprocess calls below. It streams output live so a long
run is visible while it happens, and raises on a non-zero exit rather than
letting a failed step look like a finished one.

In [ ]:
import subprocess
import time


HYDRA_QUIET = ['hydra.run.dir=.', 'hydra.output_subdir=null']


def sh(cmd, check=True):
    """Run a command, stream its output live, raise on a non-zero exit."""
    print('$', ' '.join(cmd), flush=True)
    started = time.time()
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end='')
    elapsed = time.time() - started
    print(f'\n[{elapsed / 60:.1f} min]')
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}')
    return proc.returncode

## 8. Smoke train

Twenty batches, one epoch, W&B off, into a throwaway checkpoint name. It surfaces a shape
problem, a bad `latent/z` width or an OOM in a couple of minutes.

`num_sanity_val_steps=1` is hardcoded in the script, so one validation batch runs before
any training step.

Two things to read out of it before spending the real run: the progress bar should carry
`fit/recovery/*` and `fit/spectrum/*` keys — if it does not, this checkout predates the
diagnostics and §9's kill rules will not be available — and `fit/bound/trace_cov` should
already be moving off ~1.5. There is **no `lr` here**: `LearningRateMonitor` needs a
logger and W&B is off for the smoke run, so the schedule can only be checked in §9.

In [ ]:
sh(['python', 'scripts/train/lejepa.py',
    f'profile={PROFILE}',
    f'encoder={ENCODER}',
    f'output_model_name={OUTPUT_MODEL_NAME}_smoke',
    'subdir=smoke',
    'trainer.max_epochs=1',
    # `+` because cfg.trainer is a struct: these two keys are not in lejepa.yaml
    '+trainer.limit_train_batches=20',
    '+trainer.limit_val_batches=5',
    f'loader.batch_size={BATCH_SIZE}',
    'loader.num_workers=2',
    f'program_constants.lambda={LAMBDA}',
    'wandb.enabled=false',
    *HYDRA_QUIET])

## 9. Train

Launched detached, so it survives closing the browser or the kernel; only a pod stop kills it.

### Which prefix to read, and why it is not a detail

Everything below is computed under `no_grad` and **never optimised**. That is
load-bearing: `epsilon` is only an honest measurement of the theory's bound
while it is independent of what is being minimised, and `recovery/*` scores
against `latent/z`, which is a *label*. It has always been in every batch (the
data config loads it); the forward simply never reads it for the objective.

The encoder carries BatchNorm after every conv stage plus a `BatchNorm1d`
before the head, and `LeJEPA.encode` folds the view axis into the batch — so in
**train mode `h` is a function of the whole batch**, not of one frame. Anything
reading only the marginal second moment of `h` agrees across modes; anything
reading `h[0] − h[1]` does not. Measured at epoch 25 of a real run, same
weights:

| key | `fit` | `validate` |
|---|---|---|
| `bound/trace_cov` | 9.5–9.9 | 9.87 |
| `spectrum/effective_rank` | 9.7–9.85 | 9.6 |
| `align_loss` | 0.05 | **0.11** |
| `bound/delta` | 0.02–0.3 | **2.69** |

The covariance is the same in both. `delta` differs by an order of magnitude,
and the train-mode value sat **below its own theoretical floor** — impossible
for any fixed encoder. So the whole `bound/` group is logged on the
**validation stage only**, and `align_loss` must be read against its floor
rather than against zero.

### What is logged, and what each one catches

The "step 0" column is measured, not guessed: a randomly initialised
`paper_cnn` at `n=10`, `B=256`, 224px, λ=5e-2.

| key | stage | step 0 | healthy | what a bad value means |
|---|---|---|---|---|
| `fit/loss` | both | 7.02 | falls | `λ·sigreg + (1−λ)·align`. |
| `fit/sigreg_loss` | both | 139 | **falls hard and fast** | The Epps–Pulley isotropy statistic — the only thing preventing collapse; there is no EMA target. Should drop ~two orders inside a few hundred steps. |
| `fit/align_loss` | both | 0.0697 | falls **to its floor** | `(h.mean(0) − h)²`. The floor is `(1−ρ)·trace_cov/(2n)` ≈ **0.048**, not zero. Read the `validate/` value. |
| `fit/balance/sigreg_share` | both | 0.991 | falls well below 1 | Pinned near 1 *while* `align_loss` is flat is λ too high — drop to 1e-2. Early dominance is expected and fine. |
| `fit/whitening_metric` | both | 0.0763 | drifts down | `‖Cov−I‖²_F/n²`. **Note how small it already is** — see the spectrum rows. |
| `spectrum/trace_cov` | both | 1.50 | rises toward `n`=10 | Heading to 0 is total collapse. The fastest collapse detector there is. |
| `spectrum/cov_eig_min` | both | 0.0261 | rises toward 1 | Smallest eigenvalue of `Cov(h)`. Zero is a direction carrying nothing. |
| `spectrum/effective_rank` | both | **4.69** | rises toward `n`=10 | `(Σλ)²/Σλ²` — how many directions actually carry variance. |
| `recovery/r2_z_to_h`, `r2_h_to_z` | both | 0.019 / 0.027 | rise | Linear R². Once `Cov(h)≈I` these two are **forced** to agree, so their agreement checks whitening, not recovery. |
| `recovery/procrustes_mse_per_dim` | both | 1.03 | falls | The criterion metric — but **not scale-free**: `h = 0` scores 1.0 and an isotropic uninformative `h` scores 2.0. Step 0 reads 1.03 because the embedding is *small*. |
| `recovery/procrustes_scale` | both | ~0 | rises | `Σσ_raw/n`. The part of the number above that is actually about agreement. |
| `recovery/orth_err_normalized` | both | 0.997 | falls | `‖AᵀA−I‖_F/√n` of the best linear map. |
| `recovery/cond` | both | 70.0 | falls | Condition number of that map. **Rising while `orth_err` falls** means the recovered block is tidying up while some directions stay null. |
| `recovery/recovered_dimensions` | both | ~0.2 | rises toward `n` | `n · r2_h_to_z` = Σσ². Reads as "how many latents' worth of information". |
| `validate/bound/delta`, `D`, `predicted_error`, `bound_headroom` | **validate** | — | delta falls | The theory's own prediction. `bound_headroom` is `n − predicted`: negative means the bound exceeds `E‖z‖²=n`, which `h=0` already achieves, so it is saying nothing yet. |
| `validate/bound/delta_floor` | **validate** | — | — | The floor `delta` cannot go below, `2ρ(1−ρ)·(trace_cov − recovered_dimensions)`. Compare the two **epoch means**; the per-batch boolean is not logged because at 256 pairs it false-alarms on honest batches. |
| `lr-AdamW` | — | 0 → peak | rises through warmup, then flat | From `LearningRateMonitor`, which needs a logger — absent when W&B is off. |

> **Why the spectrum rows are not redundant with `whitening_metric`.** At step 0
> the whitening metric reads 0.076, which looks like nothing — while the
> embedding lives in **4.7 of its 10 directions**. `epsilon` is a Frobenius
> aggregate and one dead direction contributes 1 to `epsilon²` out of `n`; it
> disappears into ordinary early-training values. `cov_eig_min` and
> `effective_rank` do not average it away, and partial collapse is the failure
> mode most likely to survive to the end while every loss curve looks fine.

### Kill rules — the point of all this

The realistic saving is not early-stopping a *converging* run; it is killing a
**broken** one in epoch 1 instead of at epoch 25. Check the log after ~200 steps
and again at the end of epoch 1:

1. **`lr-AdamW` is exactly 0 for the whole first epoch** → the schedule
   regressed to `interval: 'epoch'`. This has happened in this repo before and
   cost a run that looked like it trained. Kill immediately.
2. **`sigreg_loss` flat near 139, or `trace_cov` → 0** → nothing is being
   regularised, or the representation is collapsing outright. Kill.
3. **`effective_rank` below its step-0 4.69 and still falling** at the end of
   epoch 1 → SIGReg is losing. Kill; re-run at a higher λ.
4. **`recovery/r2_z_to_h` still under ~0.1 at the end of epoch 2**, with
   everything else moving → the losses are optimising and the latents are not
   being recovered. That is the one failure no loss curve shows, and it is why
   these are logged.
5. **`balance/sigreg_share` pinned at ~1 while `align_loss` is flat** → λ too
   high. Kill and re-run at 1e-2 rather than spending the budget on it.

> These diagnostics live in `wm/lejepa/losses.py` and are wired in
> `scripts/train/lejepa.py`.

In [ ]:
log_path = os.path.join(STABLEWM_HOME, 'logs', f'{SUBDIR}.log')
os.makedirs(os.path.dirname(log_path), exist_ok=True)

cmd = [
    'python', 'scripts/train/lejepa.py',
    f'profile={PROFILE}',
    f'encoder={ENCODER}',
    f'output_model_name={OUTPUT_MODEL_NAME}',
    f'subdir={SUBDIR}',
    f'seed={SEED}',
    f'trainer.max_epochs={EPOCHS}',
    f'schedule.constant_frac={CONSTANT_FRAC}',
    f'schedule.warmup_steps={WARMUP_STEPS}',
    f'loader.batch_size={BATCH_SIZE}',
    f'loader.num_workers={NUM_WORKERS}',
    f'program_constants.lambda={LAMBDA}',
    'wandb.enabled=true',
    f'wandb.config.entity={WANDB_ENTITY}',
    f'wandb.config.project={WANDB_PROJECT}',
    *HYDRA_QUIET,
]
print('$', ' '.join(cmd))

with open(log_path, 'w') as handle:
    train_proc = subprocess.Popen(
        cmd, stdout=handle, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True,  # detaches from this kernel's process group
    )

print('\nStarted PID', train_proc.pid)
print('Log:', log_path)

Poll until it exits. Interrupting this cell does **not** stop the run — it is detached. Re-running the cell resumes watching, but only while this kernel lives: after a restart `train_proc` is gone and the log tail below is the way to follow it.

In [ ]:
while train_proc.poll() is None:
    saved = sorted(Path(CKPT_DIR).glob('weights_epoch_*.pt')) if Path(CKPT_DIR).exists() else []
    tail = subprocess.run(['tail', '-n', '1', log_path], capture_output=True, text=True).stdout.strip()
    print(f'[{time.strftime("%H:%M:%S")}] {len(saved)}/{EPOCHS} epochs saved | {tail[-140:]}')
    time.sleep(120)

print('\nexit code', train_proc.returncode)

In [ ]:
!tail -n 40 "$log_path"

In [ ]:
checkpoints = sorted(
    Path(CKPT_DIR).glob('weights_epoch_*.pt'),
    key=lambda p: int(p.stem.rsplit('_', 1)[1]),
)
print(f'{len(checkpoints)} checkpoints in {CKPT_DIR}')
for path in checkpoints:
    print(' ', path.name)

encoder_hash = (Path(STABLEWM_HOME) / 'checkpoints' / SUBDIR / 'encoder_hash.txt').read_text().strip()
print(f'\nencoder hash: {encoder_hash}   ← every results row records this, so a number '
      'can be traced back to exact weights. `seed` does not make a run '
      'reproducible, so if you lose the .pt you cannot regenerate it.')

# Trainer state (optimizer + scheduler + loops), one per epoch. Its directory is
# chosen by spt.Manager, not by the config -- see §12 -- so the run records it.
pointer = Path(STABLEWM_HOME) / 'checkpoints' / SUBDIR / 'lightning_ckpt_dir.txt'
if pointer.exists():
    lightning_dir = Path(pointer.read_text().strip())
    ckpts = sorted(lightning_dir.glob('*.ckpt'))
    total = sum(c.stat().st_size for c in ckpts) / 1e9
    print(f'\ntrainer state -> {lightning_dir}')
    print(f'  {len(ckpts)} files, {total:.2f} GB: {[c.name for c in ckpts]}')
else:
    print('\nno lightning_ckpt_dir.txt — this checkout predates '
          'RecordCkptDirCallback, so the trainer state is under '
          '$SPT_CACHE_DIR/runs/<date>/<time>/<run_id>/checkpoints/')

## 10. Statistics — the frozen metric suite, on the last few checkpoints

`run_metrics.py` scores the **encoder alone**. `LeJEPA` has no dynamics by
design: the script only calls `model.encode(...)['emb']`.

Four things about how this is invoked:

- **`SCORE_LAST_N` checkpoints, not all of them.** Each scoring embeds 20k pairs
  twice over, embeds the style probe, and trains the non-linear probe. Per-epoch
  scoring was the old shape and it is not affordable; the training run already
  logs the per-step trend, and this suite is the authoritative reading taken
  where it matters.
- **BatchNorm is recalibrated first** (`RECALIBRATE_BN`). One no-grad pass with
  the weights frozen and `momentum=None`, replacing BN's ~10-batch running
  window with a cumulative average over the whole sample. Not cosmetic: under a
  constant LR the weights never settle, so a checkpoint's running statistics are
  permanently stale — and every number below is computed through them.
- **`program_constants.lambda` is passed explicitly.** `metrics.yaml` defaults
  are not guaranteed to match what the encoder trained at, and the value is
  copied *verbatim* into every row. Left alone, the rows would record a λ the
  encoder was never trained at.
- **The style probe is armed** by `metrics.yaml`'s default and resolves from
  `PROFILE`. It is what lets `delta` split into `delta_content` and
  `delta_style`; without it the bound charges style leakage to nonlinearity.

The results table is append-only. Re-running this cell adds rows rather than
replacing them, and the reader takes the last row per checkpoint.

In [ ]:
saved = sorted(
    Path(CKPT_DIR).glob('weights_epoch_*.pt'),
    key=lambda p: int(p.stem.rsplit('_', 1)[1]),
)
to_score = saved[-SCORE_LAST_N:]
print(f'{len(saved)} checkpoints on disk; scoring the last {len(to_score)}:')
for path in to_score:
    print(' ', path.name)

started = time.time()
for checkpoint in to_score:
    print(f'\n===== {checkpoint.name} =====')
    sh(['python', 'scripts/identifiability/run_metrics.py',
        f'checkpoint={OUTPUT_MODEL_NAME}/{checkpoint.name}',
        f'profile={PROFILE}',
        f'max_samples={METRIC_MAX_SAMPLES}',
        f'seed={SEED}',
        'device=cuda',
        f'recalibrate_batchnorm={str(RECALIBRATE_BN).lower()}',
        f'program_constants.lambda={LAMBDA}',
        f'results_path={RESULTS_PATH}',
        *HYDRA_QUIET])

print(f'\nscored {len(to_score)} checkpoints in {(time.time() - started) / 60:.1f} min')
print('results ->', RESULTS_PATH)

## 10b. The supervised ceiling — what was ever readable from these pixels

Every number section 10 reports is conditional on the encoder it was handed, so
a latent scoring ~0 has three explanations it cannot separate: the objective
missed it, this backbone cannot represent it, or **the renderer never put it in
the frame**. In the third case no encoder exists that could have recovered it,
and the zero is not a result about LeJEPA at all.

`run_oracle.py` re-initialises every parameter of the checkpoint and trains that
same architecture on the same pixels *with labels*. Its per-latent scores are an
upper bound on every row above:

| supervised | LeJEPA | reading |
|---|---|---|
| ~0 | ~0 | not in the pixels, or not representable by this backbone — **not a LeJEPA failure** |
| high | ~0 | the information is there and reachable; the objective did not select it |
| high | high | recovered |

Run **once**, not per checkpoint: with `init=scratch` only the checkpoint's
*architecture* is used, so the result is identical whichever one it is pointed
at. The last one is chosen purely so the row joins to the final metrics row.

In [ ]:
# The ceiling depends only on the architecture, so one run covers every row.
last = sorted(
    Path(CKPT_DIR).glob('weights_epoch_*.pt'),
    key=lambda p: int(p.stem.rsplit('_', 1)[1]),
)[-1]
print(f'architecture from {last.name}; weights re-initialised from scratch')

started = time.time()
sh(['python', 'scripts/identifiability/run_oracle.py',
    f'checkpoint={OUTPUT_MODEL_NAME}/{last.name}',
    f'profile={PROFILE}',
    'init=scratch',
    f'epochs={ORACLE_EPOCHS}',
    f'max_samples={ORACLE_MAX_SAMPLES}',
    f'seed={SEED}',
    'device=cuda',
    f'program_constants.lambda={LAMBDA}',
    f'results_path={RESULTS_PATH}',
    *HYDRA_QUIET])
print(f'\nceiling measured in {(time.time() - started) / 60:.1f} min')

## 11. Read it

Per latent first. The aggregates come second, and they are read against the
ceiling the renderer imposes rather than against zero.

In [ ]:
import numpy as np
import pandas as pd

from stable_worldmodel.identifiability import results as ident_results

rows = ident_results.load_rows(RESULTS_PATH)
table = pd.DataFrame(rows)
table = table[table.seed == SEED].sort_values('epoch').reset_index(drop=True)

last = table.iloc[-1]
print(f'{len(table)} scored checkpoints   n={last.n}   profile={last.profile}   '
      f'config_hash={last.config_hash}   suite={last.metric_suite_version}')
print(f'style probe: {bool(last.has_style_probe)}   '
      f'bn recalibrated: {last.bn_recalibrated}   '
      f'encoder hash: {last.encoder_hash}')

# ---------------------------------------------------------------- per latent
# The layer that localises a failure. `mlp - linear` separates two completely
# different findings: a latent the trained probe recovers and the linear one
# does not was FOUND and not linearised; one neither recovers was never seen.
per_latent = pd.DataFrame({
    'latent': last.probe_latent_names,
    'linear_r2': last.probe_linear_per_latent,
    'mlp_r2': last.probe_mlp_per_latent,
})
per_latent['mlp - linear'] = per_latent.mlp_r2 - per_latent.linear_r2
print(f'\n--- per-latent read-out at epoch {last.epoch} ---')
print(per_latent.sort_values('linear_r2', ascending=False).round(4).to_string(index=False))

dead = [last.probe_latent_names[i] for i in last.dead_latents]
print(f'\nunrecovered (linear R2 < 0.01): {dead or "none"}')

# ----------------------------------------------------------- the aggregates
print('\n--- aggregates, against their own ceiling ---')
print(f'  recovered_dimensions   {last.recovered_dimensions:7.3f}  of trace_cov '
      f'{last.trace_cov:.3f}, across {last.canonical_participation:.2f} effective directions')
print(f'  probe_linear_r2        {last.probe_linear_r2:7.4f}  ceiling at this '
      f'observability {last.r2_ceiling:.3f}')
print(f'  procrustes_mse_per_dim {last.procrustes_mse_per_dim:7.4f}  floor '
      f'{last.procrustes_floor:.3f} / whitened target '
      f'{last.procrustes_floor_whitened:.3f} / uninformative isotropic 2.0')
print(f'  canonical spectrum     {np.round(last.canonical_corr, 3).tolist()}')

# ------------------------------------------------------------------ the bound
print('\n--- the bound, and whether it may be quoted ---')
verdict = 'OK' if last.delta_admissible else 'IMPOSSIBLE -- do not quote this row'
print(f'  residual_hermite_degree {last.residual_hermite_degree:6.2f}   [{verdict}]')
print(f'  delta {last.delta:.4f} = content {last.delta_content:.4f} + style '
      f'{last.delta_style:.4f}   floor {last.delta_floor:.4f}')
print(f'  D {last.D:.2f}  predicted_error {last.predicted_error:.2f}  '
      f'vacuous {bool(last.bound_vacuous)}  '
      f'(non-vacuous needs mean probe R2 > {last.mean_probe_r2_needed:.3f})')

# ------------------------------------------------------------------ the trend
COLUMNS = [
    'epoch', 'procrustes_mse_per_dim', 'procrustes_floor_whitened',
    'probe_linear_r2', 'probe_mlp_r2', 'r2_ceiling',
    'recovered_dimensions', 'canonical_participation',
    'epsilon', 'trace_cov', 'cov_eig_min', 'effective_rank',
    'delta_content', 'delta_style', 'delta_floor',
    'residual_hermite_degree', 'delta_admissible',
    'predicted_error', 'bound_vacuous',
    'orth_err_normalized', 'cond', 'style_sensitivity', 'sigreg_z',
]
pd.set_option('display.width', 240, 'display.max_columns', 60)
table[[c for c in COLUMNS if c in table]].round(4)

Small multiples, one metric per panel — the honest form when the panels share
an x axis (epoch) and nothing else. Never two y-scales on one panel: the
two-series panels below pair quantities already in the same units.

With only a handful of scored checkpoints this is a sanity check on direction,
not a curve. The per-latent table above is the result.

In [ ]:
import matplotlib.pyplot as plt

INK, INK_2, MUTED = '#0b0b0b', '#52514e', '#8a8880'
SURFACE, GRIDLINE = '#fcfcfb', '#e6e5e1'
S1, S2 = '#2a78d6', '#eb6834'   # validated categorical slots 1 and 2

PANELS = [
    ('Recovery error vs its floor',
     [('procrustes_mse_per_dim', 'measured'), ('procrustes_floor_whitened', 'floor at this ceiling')],
     'not scale-free: h = 0 scores 1.0, uninformative isotropic scores 2.0.'),
    ('Probe R^2 vs its ceiling',
     [('probe_linear_r2', 'linear'), ('r2_ceiling', 'ceiling (live latents / n)')],
     'the gap to the ceiling is what training can still close.'),
    ('Linear vs non-linear probe',
     [('probe_linear_r2', 'linear'), ('probe_mlp_r2', 'trained MLP')],
     'a gap means present-but-not-linearised, not absent.'),
    ('Recovered dimensions',
     [('recovered_dimensions', 'sum sigma^2'), ('canonical_participation', 'effective directions')],
     'how many latents-worth, and spread over how many directions.'),
    ('delta vs its floor',
     [('delta_content', 'content'), ('delta_floor', 'floor')],
     'content below floor is impossible -- the row was not measured in eval mode.'),
    ('Residual Hermite degree', [('residual_hermite_degree', None)],
     'below 2 is impossible. Read this before the bound.'),
    ('epsilon = ||Cov(h) - I||_F', [('epsilon', None)],
     "the bound's first term. Logged during training, never optimised."),
    ('Spectrum floor', [('cov_eig_min', None)],
     'smallest eigenvalue of Cov(h). Partial collapse hides from epsilon.'),
    ('Effective rank', [('effective_rank', None)],
     '(sum lambda)^2 / sum lambda^2. Toward n.'),
    ('Condition number', [('cond', None)],
     'of the best linear map. Rising while orth_err falls = null directions.'),
    ('Style sensitivity', [('style_sensitivity', None)],
     'what the alignment loss is supposed to discard. Down.'),
    ('SIGReg z-score', [('sigreg_z', None)],
     'against a matched i.i.d.-Gaussian null. Cov = I is not isotropy.'),
]

available = [p for p in PANELS if all(k in table and table[k].notna().any() for k, _ in p[1])]
skipped = [p[0] for p in PANELS if p not in available]

epochs = table['epoch'].to_numpy()
ncols = 3
nrows = -(-len(available) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.5 * nrows), dpi=130)
fig.patch.set_facecolor(SURFACE)

for ax, (title, series, note) in zip(axes.ravel(), available):
    ax.set_facecolor(SURFACE)
    for (key, label), color in zip(series, (S1, S2)):
        y = table[key].to_numpy(dtype=float)
        ax.plot(epochs, y, color=color, lw=1.8, marker='o', ms=5.5,
                mfc=color, mec=SURFACE, mew=1.2, label=label, zorder=3)
        ax.annotate(f'{y[-1]:.3g}', (epochs[-1], y[-1]), textcoords='offset points',
                    xytext=(7, 0), va='center', fontsize=8.5, color=INK_2)

    ax.set_title(title, fontsize=11, color=INK, loc='left', pad=24)
    ax.text(0.0, 1.035, note, transform=ax.transAxes, fontsize=8.5,
            color=MUTED, ha='left', va='bottom')
    ax.grid(axis='y', color=GRIDLINE, lw=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRIDLINE)
    ax.tick_params(colors=MUTED, labelsize=9, length=3)
    ax.set_xticks(epochs)
    span = max(epochs[-1] - epochs[0], 1)
    ax.set_xlim(epochs[0] - 0.3, epochs[-1] + 0.20 * span)
    if len(series) > 1:
        ax.legend(frameon=False, fontsize=8.5, labelcolor=INK_2, loc='best')

for ax in axes.ravel()[len(available):]:
    ax.set_visible(False)
for ax in axes.ravel()[max(0, len(available) - ncols):len(available)]:
    ax.set_xlabel('epoch', fontsize=9, color=INK_2)

fig.suptitle(
    f'LeJEPA encoder recovery — {PROFILE} (n={table.n.iloc[0]}), '
    f'{ENCODER}, seed {SEED}, one run',
    fontsize=12.5, color=INK, x=0.012, ha='left', y=0.997,
)
fig.tight_layout(rect=(0, 0, 1, 0.985))

plot_path = os.path.join(OUTPUT_DIR, f'lejepa_trend_{SUBDIR}.png')
fig.savefig(plot_path, dpi=160, facecolor=SURFACE, bbox_inches='tight')
plt.show()

if skipped:
    print('panels skipped (metric absent or all-NaN):', ', '.join(skipped))
print('saved ->', plot_path)

## 12. How to read it, and what is on disk

Most of the *shape* was already visible during training (§9). `run_metrics.py`
differs in three ways that matter: it scores the full eval set rather than one
batch, it scores in **eval mode with refreshed BatchNorm statistics**, and it
reports **per latent**. If §9's kill rules were watched, nothing here should be
a surprise; if something is, that gap is itself worth recording.

**Read it in this order.**

1. **The per-latent table.** A mean over ten latents whose observability through
   the renderer spans two orders of magnitude is not a number about the encoder.
   Which coordinates came in, and which did not, is the finding.
2. **`residual_hermite_degree`.** Below 2 the row is *impossible* — the
   embeddings were not a function of one frame — and nothing in the bound group
   may be quoted from it. Check this before reading `predicted_error`.
3. **The aggregates against their ceilings.** `probe_linear_r2` against
   `r2_ceiling`, `procrustes_mse_per_dim` against `procrustes_floor_whitened`.
   Against zero they are meaningless.
4. **`bound_vacuous` and `mean_probe_r2_needed`.** Below that requirement, a
   vacuous bound is an arithmetic certainty, not a disappointing result.

**The shapes that mean something is wrong.**

- `delta_admissible` false → the row was not produced in eval mode. Not a bad
  encoder; a bad measurement.
- `procrustes_mse_per_dim` near 1.0 and flat with `trace_cov` small → the
  embedding is shrinking rather than recovering. `h = 0` scores 1.0.
- `effective_rank` well below `n` while `epsilon` looks fine → partial collapse,
  which the aggregate hides by construction.
- `cond` rising while `orth_err_normalized` falls → the recovered block is
  tidying up while some directions stay null. Those latents are flat directions
  in the embedding.
- A large `mlp − linear` gap on a latent → the encoder found that quantity and
  did not linearise it. A completely different finding from never seeing it.

**Two known ceilings, not encoder failures.** The camera is a pick-and-place
frame and cannot be reframed, so `cube.pos_xy[0]` keeps ~27 px of travel against
`[1]`'s 96 px, and `cube.pos_z` less than both. And `cube.yaw`'s pixel
separability is **not monotone in angle** — measured rms 3.79 at 21°, 4.36 at
43°, 1.17 at 86°, because 90° *is* the identification — so the sampled arc's two
endpoints are the least separable pair in it. Score those with a stated ceiling
rather than letting them drag the aggregate. Both are properties of `g`, not of
the encoder.

```
$STABLEWM_HOME/
├── checkpoints/
│   ├── lejepa/weights_epoch_{1..N}.pt    ← one per epoch, never pruned
│   ├── lejepa/config.json                ← cfg.model, what load_pretrained reads
│   └── lejepa_s3072/
│       ├── config.yaml, encoder_hash.txt
│       ├── lightning_ckpt_dir.txt        ← where the trainer state really went
│       └── lightning -> $SPT_CACHE_DIR/runs/<date>/<time>/<run_id>/checkpoints/
├── outputs/
│   └── lejepa_results_lejepa_s3072.jsonl ← append-only, one row per scoring
└── logs/lejepa_s3072.log
```

### Two kinds of checkpoint, and only one has a settable path

| | `weights_epoch_N.pt` | `epochNNN.ckpt` |
|---|---|---|
| written by | `SaveCkptCallback` | `ModelCheckpoint` (`checkpoint.keep_every_epoch`) |
| contains | weights only | weights **+** optimizer, scheduler, loops, RNG |
| epoch index | from **1** | from **0** — `epoch004.ckpt` produced `weights_epoch_5.pt` |
| used for | `load_pretrained`, and §10's scoring | resuming a killed run |
| path | `$STABLEWM_HOME/checkpoints/$OUTPUT_MODEL_NAME/` — yours | **not settable** |
| size | ~11 MB (`paper_cnn`) | ~34 MB (`paper_cnn`), ~270 MB (`vit_small`) |

**Why the second path is not settable.** `spt.Manager` always runs in cache_dir
mode — a cache dir is mandatory, passing `None` raises — and
`_configure_cache_dir_checkpointing` rewrites the `dirpath` of *every*
`ModelCheckpoint` to `$SPT_CACHE_DIR/runs/<date>/<time>/<run_id>/checkpoints/`.
What you *can* set is `SPT_CACHE_DIR` (cell 1). The `<date>/<time>/<run_id>`
tail is not knowable before the run starts, so `RecordCkptDirCallback` resolves
it at **train start** and writes `lightning_ckpt_dir.txt` next to
`config.yaml` — at train start rather than after `fit` returns, because the run
whose checkpoint directory you need to find is precisely the run whose `fit`
never returns.

> **Resuming is not wired.** `lejepa.py` passes `ckpt_path=None`, and two traps
> sit behind changing that: `spt.Manager`'s `weights_only` defaults to **True**
> (optimizer and scheduler silently discarded — transfer-init, not resume), and
> Manager auto-resumes from its own `last.ckpt` only when
> `SLURM_RESTART_COUNT >= 1`, which never happens on RunPod. To actually
> resume, pass the absolute `last.ckpt` path *and* `weights_only=False`.

### What to run next

The two questions this measurement leaves open, and the cheapest experiment for
each:

```bash
# 1. Is a missing latent absent from the pixels, or absent from the objective?
#    Train the SAME encoder supervised on latent/z. If it also reads ~0 on a
#    coordinate, the ceiling is g plus this architecture and no amount of
#    LeJEPA training moves it. If it recovers it, the objective is not selecting
#    it -- which is the interesting finding.

# 2. Does the isotropy constraint force filler?  With m = n the encoder must
#    fill n unit-variance directions even when g exposes fewer; ask for fewer
#    and delta should collapse. recovery_diagnostics returns nothing under
#    m != n, so score it on the per-latent probe alone.
python scripts/train/lejepa.py model.head.output_dim=5 subdir=lejepa_m5
```